In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t036'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')

alert_title, alert_window, alert_entity

```
Out[1]:
```
The alert is titled "cart响应时间突增告警" (Cart Response Time Sudden Increase Alert). It occurred between 2026-04-13 04:22:01 and 2026-04-13 04:28:03 (UTC+8). The entity that triggered the alert is the operation `cart::POST /oteldemo.CartService/AddItem` of type `apm.operation` in the `apm` domain.

The original code execution output of IPython Kernel is also provided below for reference:

('cart响应时间突增告警', {'start': '2026-04-13T04:22:01+08:00', 'end': '2026-04-13T04:28:03+08:00'}, {'entity_id': 'e433b3f842dc4e525a834701cc603c28', 'entity_name': 'cart::POST /oteldemo.CartService/AddItem', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# Filter metrics for the 'cart' service and its operation
cart_service_metrics = metrics_data[
    (metrics_data['entity_name'].str.contains('cart', case=False, na=False))
]

# List all unique KPIs (metrics) for the 'cart' service and its operation
cart_kpis = cart_service_metrics['metric'].unique()
cart_kpis

```
Out[2]:
```
The available KPIs for the `cart` service and its operation are: `request_count`, `avg_request_latency_seconds`, `cpu`, `latency`, `mem`, and `workload`. These KPIs can be analyzed further.

The original code execution output of IPython Kernel is also provided below for reference:

['request_count' 'avg_request_latency_seconds' 'cpu' 'latency' 'mem'
 'workload']```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Convert time column to numeric for calculations
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Calculate global P95 thresholds for each KPI
global_p95_thresholds = cart_service_metrics.groupby('metric')['value'].quantile(0.95)

# Filter data for the alert time window
alert_start = datetime.fromisoformat('2026-04-13T04:22:01+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-13T04:28:03+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_start_microseconds = int(alert_start.timestamp() * 1_000_000)
alert_end_microseconds = int(alert_end.timestamp() * 1_000_000)

filtered_cart_metrics = cart_service_metrics[
    (cart_service_metrics['time'] >= alert_start_microseconds) &
    (cart_service_metrics['time'] <= alert_end_microseconds)
]

global_p95_thresholds, filtered_cart_metrics

```
Out[3]:
```
The global P95 thresholds for the `cart` service KPIs are as follows:
- `avg_request_latency_seconds`: 0.03688 seconds
- `cpu`: 0.07709
- `latency`: 0.02113
- `mem`: 117,605,700 bytes
- `request_count`: 3,928.3
- `workload`: 9,354.6

During the alert time window (2026-04-13 04:22:01 to 2026-04-13 04:28:03, UTC+8), 260 metric records were filtered for the `cart` service and its operation. These records will be analyzed further to identify anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(metric
avg_request_latency_seconds    3.688025e-02
cpu                            7.709009e-02
latency                        2.112848e-02
mem                            1.176057e+08
request_count                  3.928300e+03
workload                       9.354600e+03
Name: value, dtype: float64,                    time domain          entity_set                         entity_id                               entity_name         metric         value                            metric_set_id service
56681  1776025321000000    apm       apm.operation  7bb1b1512922edfdefac66f31c640a81  cart::POST /oteldemo.CartService/GetCart  request_count  4.556000e+03  apm@metric_set@apm.metric.apm.operation    None
56682  1776025351000000    apm       apm.operation  7bb1b1512922edfdefac66f31c640a81  cart::POST /oteldemo.CartService/GetCart  request_count  3.908000e+03  apm@metric_set@apm.metric.apm.operation    None
56683  1776025381000000    apm       apm.operation  7bb1b1512922edfdefac66f31c640a81  cart::POST /oteldemo.CartService/GetCart  request_count  2.589000e+03  apm@metric_set@apm.metric.apm.operation    None
56684  1776025411000000    apm       apm.operation  7bb1b1512922edfdefac66f31c640a81  cart::POST /oteldemo.CartService/GetCart  request_count  2.717000e+03  apm@metric_set@apm.metric.apm.operation    None
56685  1776025441000000    apm       apm.operation  7bb1b1512922edfdefac66f31c640a81  cart::POST /oteldemo.CartService/GetCart  request_count  3.182000e+03  apm@metric_set@apm.metric.apm.operation    None
...                 ...    ...                 ...                               ...                                       ...            ...           ...                                      ...     ...
84286  1776025651000000    apm  apm.service.legacy                                                                        cart       workload  4.889000e+03                                     None    None
84353  1776025681000000    apm  apm.service.legacy                                                                        cart            cpu  4.998892e-02                                     None    None
84354  1776025681000000    apm  apm.service.legacy                                                                        cart        latency  2.117211e-02                                     None    None
84355  1776025681000000    apm  apm.service.legacy                                                                        cart            mem  1.163919e+08                                     None    None
84356  1776025681000000    apm  apm.service.legacy                                                                        cart       workload  3.997000e+03                                     None    None

[260 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Identify anomalies by checking if any data points exceed their respective global P95 thresholds
anomalies = filtered_cart_metrics[
    filtered_cart_metrics.apply(
        lambda row: row['value'] > global_p95_thresholds.get(row['metric'], float('inf')), axis=1
    )
]

# Select relevant columns for anomaly reporting
anomalies_report = anomalies[['time', 'metric', 'value']].sort_values(by=['metric', 'time'])

anomalies_report

```
Out[4]:
```
During the alert time window, the following anomalies were identified for the `cart` service and its operation:

1. **`avg_request_latency_seconds`**: Multiple data points exceeded the global P95 threshold of 0.03688 seconds, with values ranging up to 0.109 seconds.

2. **`cpu`**: One data point exceeded the global P95 threshold of 0.07709, with a value of 0.079.

3. **`latency`**: Four data points exceeded the global P95 threshold of 0.02113, with values up to 0.02417.

4. **`request_count`**: Two data points exceeded the global P95 threshold of 3928.3, with values of 4556 and 4551.

These anomalies indicate significant deviations in response latency, CPU usage, and request count during the alert period, which require further investigation.

The original code execution output of IPython Kernel is also provided below for reference:

time                       metric        value
63506  1776025381000000  avg_request_latency_seconds     0.050774
72892  1776025381000000  avg_request_latency_seconds     0.052232
63507  1776025411000000  avg_request_latency_seconds     0.055851
72893  1776025411000000  avg_request_latency_seconds     0.054584
61443  1776025441000000  avg_request_latency_seconds     0.043692
63508  1776025441000000  avg_request_latency_seconds     0.072517
72894  1776025441000000  avg_request_latency_seconds     0.053269
61444  1776025471000000  avg_request_latency_seconds     0.058775
63509  1776025471000000  avg_request_latency_seconds     0.097090
64043  1776025471000000  avg_request_latency_seconds     0.037693
72895  1776025471000000  avg_request_latency_seconds     0.052955
61445  1776025501000000  avg_request_latency_seconds     0.066298
63510  1776025501000000  avg_request_latency_seconds     0.108434
64044  1776025501000000  avg_request_latency_seconds     0.039061
72896  1776025501000000  avg_request_latency_seconds     0.053474
61446  1776025531000000  avg_request_latency_seconds     0.066145
63511  1776025531000000  avg_request_latency_seconds     0.109054
64045  1776025531000000  avg_request_latency_seconds     0.040804
72897  1776025531000000  avg_request_latency_seconds     0.053150
61447  1776025561000000  avg_request_latency_seconds     0.052319
63512  1776025561000000  avg_request_latency_seconds     0.086284
64046  1776025561000000  avg_request_latency_seconds     0.041824
72898  1776025561000000  avg_request_latency_seconds     0.053074
61448  1776025591000000  avg_request_latency_seconds     0.042460
63513  1776025591000000  avg_request_latency_seconds     0.069374
64047  1776025591000000  avg_request_latency_seconds     0.040633
72899  1776025591000000  avg_request_latency_seconds     0.051369
61449  1776025621000000  avg_request_latency_seconds     0.050255
63514  1776025621000000  avg_request_latency_seconds     0.082677
64048  1776025621000000  avg_request_latency_seconds     0.040235
72900  1776025621000000  avg_request_latency_seconds     0.052173
61450  1776025651000000  avg_request_latency_seconds     0.050507
63515  1776025651000000  avg_request_latency_seconds     0.083213
64049  1776025651000000  avg_request_latency_seconds     0.040367
72901  1776025651000000  avg_request_latency_seconds     0.053015
61451  1776025681000000  avg_request_latency_seconds     0.056945
63516  1776025681000000  avg_request_latency_seconds     0.094208
64050  1776025681000000  avg_request_latency_seconds     0.039610
72902  1776025681000000  avg_request_latency_seconds     0.052755
83513  1776025321000000                          cpu     0.079001
83864  1776025471000000                      latency     0.021593
83934  1776025501000000                      latency     0.024154
84004  1776025531000000                      latency     0.024174
84354  1776025681000000                      latency     0.021172
56681  1776025321000000                request_count  4556.000000
65873  1776025321000000                request_count  4551.000000```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Convert time columns to numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Filter traces for the `cart::POST /oteldemo.CartService/AddItem` operation during the alert time window
filtered_traces = traces_data[
    (traces_data['spanName'] == 'cart::POST /oteldemo.CartService/AddItem') &
    (traces_data['startTime'] >= alert_start_microseconds * 1000) &
    (traces_data['endTime'] <= alert_end_microseconds * 1000)
]

# Identify traces with errors or high latency
error_traces = filtered_traces[
    (filtered_traces['statusCode'] != 0) | (filtered_traces['duration'] > global_p95_thresholds['latency'] * 1_000_000_000)
]

# Analyze the call chain to determine the most downstream faulty service
downstream_faulty_services = error_traces[['traceId', 'spanName', 'serviceName', 'statusCode', 'duration']].sort_values(by='duration', ascending=False)

downstream_faulty_services

```
Out[5]:
```
No traces with errors or high latency were found for the `cart::POST /oteldemo.CartService/AddItem` operation during the alert time window. This suggests that the anomalies observed in the metrics may not have propagated downstream or caused traceable errors in the call chain. Further investigation into other potential causes is needed.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanName, serviceName, statusCode, duration]
Index: []```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Convert the log time column to numeric for filtering
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs for the `cart` service during the alert time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start_microseconds) &
    (logs_data['_time_'] <= alert_end_microseconds) &
    (logs_data['_container_name_'].str.contains('cart', case=False, na=False))
]

# Search for error messages or unusual patterns in the log content
error_logs = filtered_logs[filtered_logs['content'].str.contains('error|exception|fail|critical', case=False, na=False)]

error_logs

```
Out[6]:
```
No error messages or unusual patterns were found in the logs for the `cart` service during the alert time window. This indicates that the anomalies observed in the metrics are not directly reflected in the service logs. Further investigation into other potential sources of the issue is required.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, __topic__, __tag__:__pack_id__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[7]:

# Filter metrics for the `k8s.node` entity set
k8s_node_metrics = metrics_data[metrics_data['entity_set'] == 'k8s.node']

# Identify all available KPIs for the `k8s.node` entity set
k8s_node_kpis = k8s_node_metrics['metric'].unique()

# Calculate global P95 thresholds for these KPIs
k8s_node_p95_thresholds = k8s_node_metrics.groupby('metric')['value'].quantile(0.95)

# Filter the data within the alert time window
filtered_k8s_node_metrics = k8s_node_metrics[
    (k8s_node_metrics['time'] >= alert_start_microseconds) &
    (k8s_node_metrics['time'] <= alert_end_microseconds)
]

k8s_node_kpis, k8s_node_p95_thresholds, filtered_k8s_node_metrics

```
Out[7]:
```
The available KPIs for the `k8s.node` entity set are:
- `node_not_ready_status`
- `node_ready_status`
- `node_memory_usage_rate`
- `node_disk_usage_rate`
- `node_pod_running_count`
- `node_cpu_usage_rate`

The global P95 thresholds for these KPIs are:
- `node_cpu_usage_rate`: 25.55
- `node_disk_usage_rate`: 36.25
- `node_memory_usage_rate`: 90.14
- `node_not_ready_status`: 0.0
- `node_pod_running_count`: 773.0
- `node_ready_status`: 1.0

During the alert time window (2026-04-13 04:22:01 to 2026-04-13 04:28:03, UTC+8), 1,131 metric records were filtered for the `k8s.node` entity set. These records will be analyzed further to identify anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(array(['node_not_ready_status', 'node_ready_status',
       'node_memory_usage_rate', 'node_disk_usage_rate',
       'node_pod_running_count', 'node_cpu_usage_rate'], dtype=object), metric
node_cpu_usage_rate        25.550804
node_disk_usage_rate       36.252482
node_memory_usage_rate     90.137460
node_not_ready_status       0.000000
node_pod_running_count    773.000000
node_ready_status           1.000000
Name: value, dtype: float64,                    time domain entity_set entity_id entity_name                 metric      value                                     metric_set_id service
120    1776025321000000    k8s   k8s.node                        node_not_ready_status   0.000000  k8s@metric_set@k8s.metric.high_level_metric_node    None
121    1776025351000000    k8s   k8s.node                        node_not_ready_status   0.000000  k8s@metric_set@k8s.metric.high_level_metric_node    None
122    1776025381000000    k8s   k8s.node                        node_not_ready_status   0.000000  k8s@metric_set@k8s.metric.high_level_metric_node    None
123    1776025411000000    k8s   k8s.node                        node_not_ready_status   0.000000  k8s@metric_set@k8s.metric.high_level_metric_node    None
124    1776025441000000    k8s   k8s.node                        node_not_ready_status   0.000000  k8s@metric_set@k8s.metric.high_level_metric_node    None
...                 ...    ...        ...       ...         ...                    ...        ...                                               ...     ...
11910  1776025561000000    k8s   k8s.node                          node_cpu_usage_rate  79.265769  k8s@metric_set@k8s.metric.high_level_metric_node    None
11911  1776025591000000    k8s   k8s.node                          node_cpu_usage_rate  90.011052  k8s@metric_set@k8s.metric.high_level_metric_node    None
11912  1776025621000000    k8s   k8s.node                          node_cpu_usage_rate  99.997839  k8s@metric_set@k8s.metric.high_level_metric_node    None
11913  1776025651000000    k8s   k8s.node                          node_cpu_usage_rate  99.998765  k8s@metric_set@k8s.metric.high_level_metric_node    None
11914  1776025681000000    k8s   k8s.node                          node_cpu_usage_rate  99.998766  k8s@metric_set@k8s.metric.high_level_metric_node    None

[1131 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

# Identify anomalies by checking if any data points exceed their respective global P95 thresholds
k8s_node_anomalies = filtered_k8s_node_metrics[
    filtered_k8s_node_metrics.apply(
        lambda row: row['value'] > k8s_node_p95_thresholds.get(row['metric'], float('inf')), axis=1
    )
]

# Select relevant columns for anomaly reporting
k8s_node_anomalies_report = k8s_node_anomalies[['time', 'entity_name', 'metric', 'value']].sort_values(by=['metric', 'time'])

k8s_node_anomalies_report

```
Out[8]:
```
During the alert time window, the following anomalies were identified for the `k8s.node` entity set:

1. **`node_cpu_usage_rate`**: Multiple data points exceeded the global P95 threshold of 25.55, with values ranging up to 99.9988.

2. **`node_disk_usage_rate`**: Multiple data points exceeded the global P95 threshold of 36.25, with values ranging up to 36.2824.

3. **`node_pod_running_count`**: Several data points exceeded the global P95 threshold of 773, with values up to 776.

These anomalies indicate high CPU usage, disk usage, and an increased number of running pods on the node during the alert period, which may have contributed to the observed issues. Further investigation into node-level resource contention is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

time entity_name                  metric       value
11906  1776025441000000                 node_cpu_usage_rate   36.086139
11907  1776025471000000                 node_cpu_usage_rate   46.813416
11908  1776025501000000                 node_cpu_usage_rate   57.648488
11909  1776025531000000                 node_cpu_usage_rate   68.454120
11910  1776025561000000                 node_cpu_usage_rate   79.265769
11911  1776025591000000                 node_cpu_usage_rate   90.011052
11912  1776025621000000                 node_cpu_usage_rate   99.997839
11913  1776025651000000                 node_cpu_usage_rate   99.998765
11914  1776025681000000                 node_cpu_usage_rate   99.998766
7381   1776025321000000                node_disk_usage_rate   36.272926
7382   1776025351000000                node_disk_usage_rate   36.279814
7383   1776025381000000                node_disk_usage_rate   36.279947
7384   1776025411000000                node_disk_usage_rate   36.280401
7385   1776025441000000                node_disk_usage_rate   36.280678
7386   1776025471000000                node_disk_usage_rate   36.280963
7387   1776025501000000                node_disk_usage_rate   36.281158
7388   1776025531000000                node_disk_usage_rate   36.281140
7389   1776025561000000                node_disk_usage_rate   36.281408
7390   1776025591000000                node_disk_usage_rate   36.281626
7391   1776025621000000                node_disk_usage_rate   36.281913
7392   1776025651000000                node_disk_usage_rate   36.282201
7393   1776025681000000                node_disk_usage_rate   36.282415
9299   1776025321000000              node_pod_running_count  774.000000
9306   1776025531000000              node_pod_running_count  776.000000
9307   1776025561000000              node_pod_running_count  776.000000
9308   1776025591000000              node_pod_running_count  776.000000
9309   1776025621000000              node_pod_running_count  776.000000
9310   1776025651000000              node_pod_running_count  776.000000```
```